<a href="https://colab.research.google.com/github/lanaajs/Processamento-de-Linguagem-Natural-Python/blob/Aulas/03_Aula.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words("portuguese"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
dados = {
    "id_bug": ["BUG-01", "BUG-02", "BUG-03", "BUG-04", "BUG-05", "BUG-06", "BUG-07"],
    "descricao": [
        "O aplicativo fecha sozinho quando tento fazer login com a conta do Google.",
        "Erro de timeout ao tentar conectar no banco de dados de produção.",
        "Botão de finalizar compra está quebrado na tela do carrinho.",
        "A tela de login congela ao usar autenticação em duas etapas.",
        "Lentidão extrema na query de relatórios de vendas do mês.",
        "O carrinho de compras esvazia sozinho após atualizar a página.",
        "Falha de conexão com a API de pagamentos via PIX."
    ],
    "status": ["Resolvido", "Em Análise", "Resolvido", "Aberto", "Resolvido", "Aberto", "Em Análise"]
}

In [ ]:
df_bugs = pd.DataFrame(dados)
print("Base carregada com sucesso!")
display(df_bugs.head())

Base carregada com sucesso!


,id_bug,descricao,status
0,BUG-01,O aplicativo fecha sozinho quando tento fazer ...,Resolvido
1,BUG-02,Erro de timeout ao tentar conectar no banco de...,Em Análise
2,BUG-03,Botão de finalizar compra está quebrado na tel...,Resolvido
3,BUG-04,A tela de login congela ao usar autenticação e...,Aberto
4,BUG-05,Lentidão extrema na query de relatórios de ven...,Resolvido


In [ ]:
def processar_texto(texto):
  texto = texto.lower()
  texto = re.sub(r"[^a-záéíóúãõâêîôûç\s]", "", texto)

  palavras = texto.split()
  palavras_limpas = [p for p in palavras if p not in stop_words]

  return " ".join(palavras_limpas)

In [ ]:
df_bugs['texto_limpo'] = df_bugs['descricao'].apply(processar_texto)

print("\n Comparação: original vs limpo")
display(df_bugs[['descricao', 'texto_limpo']])


 Comparação: original vs limpo


,descricao,texto_limpo
0,O aplicativo fecha sozinho quando tento fazer ...,aplicativo fecha sozinho tento fazer login con...
1,Erro de timeout ao tentar conectar no banco de...,erro timeout tentar conectar banco dados produção
2,Botão de finalizar compra está quebrado na tel...,botão finalizar compra quebrado tela carrinho
3,A tela de login congela ao usar autenticação e...,tela login congela usar autenticação duas etapas
4,Lentidão extrema na query de relatórios de ven...,lentidão extrema query relatórios vendas mês
5,O carrinho de compras esvazia sozinho após atu...,carrinho compras esvazia sozinho após atualiza...
6,Falha de conexão com a API de pagamentos via PIX.,falha conexão api pagamentos via pix


In [ ]:
vetorizador = TfidfVectorizer(ngram_range=(1, 2))
matriz_tfidf = vetorizador.fit_transform(df_bugs['texto_limpo'])

vocabulario = vetorizador.get_feature_names_out()
print(f"\n O modelo extraiu {len(vocabulario)} características (features) únicas.")
print("Exemplos de features aprendidas: ", vocabulario[10: 20])


 O modelo extraiu 83 características (features) únicas.
Exemplos de features aprendidas:  ['banco' 'banco dados' 'botão' 'botão finalizar' 'carrinho'
 'carrinho compras' 'compra' 'compra quebrado' 'compras' 'compras esvazia']


In [ ]:
def checar_duplicidade(novo_bug, limiar_alerta=0.15):
  novo_bug_limpo = processar_texto(novo_bug)
  vetor_novo = vetorizador.transform([novo_bug_limpo])

  similaridade = cosine_similarity(vetor_novo, matriz_tfidf).flatten()

  indice_maior = np.argmax(similaridade)
  pontuacao = similaridade[indice_maior]

  print("\n" + "="*50)
  print(f"REOPORT > '{novo_bug}'")

  if pontuacao >= limiar_alerta:
    print(f"Possível bug duplicado! ({pontuacao*100}%) de similaridade. ")
    print(f"-> Bug Original [{df_bugs['id_bug'][indice_maior]} - {df_bugs['status'][indice_maior]}]:")
    print(f"{df_bugs['descricao'][indice_maior]}")
  else:
    print("Nenhum bug similar encontrado. Abrindo novo ticket no SISTEMA!")
    print("="*50)

In [ ]:
print("\n SISTEMA DE TRIAGEM DE BUGS INICIADO. Digite 'sair' para encerrar. ")

while True:
  entrada_usuario = input("\n Descreva o problema encontrado: ")
  if entrada_usuario.lower() == 'sair':
    print("Sistema encerrado.")
    break

  checar_duplicidade(entrada_usuario)



 SISTEMA DE TRIAGEM DE BUGS INICIADO. Digite 'sair' para encerrar. 

 Descreva o problema encontrado: o app fechou sozinho

REOPORT > 'o app fechou sozinho'
Possível bug duplicado! (23.593713062974604%) de similaridade. 
-> Bug Original [BUG-06 - Aberto]:
O carrinho de compras esvazia sozinho após atualizar a página.

 Descreva o problema encontrado: o aplicativo fechou sozinho

REOPORT > 'o aplicativo fechou sozinho'
Possível bug duplicado! (34.274392984916425%) de similaridade. 
-> Bug Original [BUG-01 - Resolvido]:
O aplicativo fecha sozinho quando tento fazer login com a conta do Google.

 Descreva o problema encontrado: O aplicativo fecha sozinho quando tento fazer login com a conta do Google.

REOPORT > 'O aplicativo fecha sozinho quando tento fazer login com a conta do Google.'
Possível bug duplicado! (100.0%) de similaridade. 
-> Bug Original [BUG-01 - Resolvido]:
O aplicativo fecha sozinho quando tento fazer login com a conta do Google.
